# Gender Representation in German Plays: Fetch Data

> **Authors:** Sandra Densch-Glazov, Leonie Wichers, Kyung Yun Choi, Benedikt Schuh  
> **Contributors:** Ingo Börner (University of Potsdam, Germany), <a href="https://orcid.org/0000-0001-8294-2541"><img src="https://orcid.org/assets/vectors/orcid.logo.icon.svg" alt="ORCID iD" width="16" height="16" style="vertical-align: -2px"/> 0000-0001-8294-2541</a>: update to PyDraCor 3 and the current DraCor API v1 (2026)  
> **Version:** 2 – PyDraCor 3, DraCor API v1 (November 2026); see the version history in the README  
> **Status:** [![Status on dracor.org](https://img.shields.io/endpoint?url=https%3A%2F%2Fraw.githubusercontent.com%2Fdracor-org%2Fdracor-notebooks%2Fmain%2Fstatus%2Fbadges%2Fgender-visualization.production.json)](https://dracor-org.github.io/dracor-notebooks/appendix/tested-environments.html) [![Status on staging](https://img.shields.io/endpoint?url=https%3A%2F%2Fraw.githubusercontent.com%2Fdracor-org%2Fdracor-notebooks%2Fmain%2Fstatus%2Fbadges%2Fgender-visualization.staging.json)](https://dracor-org.github.io/dracor-notebooks/appendix/tested-environments.html)  
> **Tested with:** DraCor API 1.4.1 (production server) and 1.5.0-rc.1 (staging server), PyDraCor 3.0.2, Python 3.12, 2026-11-17

This notebook downloads the data for `relationship_network.ipynb` from the DraCor API with [PyDraCor](https://pypi.org/project/pydracor/): for each play of the German Drama Corpus (GerDraCor) that has encoded relations between characters, the relations and the co-presence network as GraphML, and the metadata of these plays as CSV. The files are written to the folder `data`.

For all plays of GerDraCor, the download takes about 20–25 minutes, because each play is requested from the API. To try the notebook with a few plays first, set `max_plays` to a small number, e.g. `max_plays = 20`.

In [1]:
import os

import pandas as pd
from pydracor import DraCorAPI, DownloadFormat
from pydracor_base.exceptions import NotFoundException

# DraCor API v1; to use another server, e.g. the staging server, change api_base
api_base = "https://dracor.org/api/v1"
dracor = DraCorAPI(host=api_base)

# number of plays to check; None checks all plays of the corpus
max_plays = None

In [2]:
german_corpus = dracor.get_corpus("ger")
german_plays = [play.name for play in german_corpus.get_metadata()][:max_plays]

if not os.path.exists('data/relations'):
    os.makedirs('data/relations')

if not os.path.exists('data/cooccurence'):
    os.makedirs('data/cooccurence')

for play_name in german_plays:
    play = german_corpus.get_play(play_name)
    play_id = play.id
    filename = f"data/relations/{play_id}.graphml"
    try:
        relations_graphml = play.get_relations(DownloadFormat.graphml)
    except NotFoundException:  # no relations encoded for this play
        continue
    # networkX doesn't support mix of directed+undirected Graphs & nx_altair's arrows look broken
    # workaround: make graph undirected
    relations_graphml = relations_graphml.replace('directed="true"', 'directed="false"')
    with open(filename, 'w', encoding='utf-8') as file:
        file.write(relations_graphml)
    filename = f"data/cooccurence/{play_id}.graphml"
    cooccurence_graphml = play.get_networkdata(DownloadFormat.graphml)
    with open(filename, 'w', encoding='utf-8') as file:
        file.write(cooccurence_graphml)

In [2]:
german_metadata = pd.DataFrame([play.model_dump() for play in german_corpus.get_metadata()])
existing_relations_files = os.listdir('data/relations')
existing_relations_play_ids = [filename.split('.')[0] for filename in existing_relations_files]
german_metadata_filtered = german_metadata[german_metadata['id'].isin(existing_relations_play_ids)]
german_metadata_filtered.to_csv('data/german_metadata.csv', index=False, encoding='utf-8')